# 第 11 週 實作｜弧長與旋轉曲面

同一個心法的第三次應用,但這次「一片」是一小段曲線——而它的長度只需要國中的畢氏定理。今天會推出球面積 $4\pi R^{2}$,並看見阿基米德兩千年前的洞見。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜折線真的收斂到弧長嗎

觀念 1 說折線低估、$n$ 越大越準。這格把它算出來,和公式解對照——順便看樓梯逼近為什麼<strong>不會</strong>收斂。


In [ ]:
f  = lambda t: t**2
df = lambda t: 2*t
a, b = 0.0, 1.0
exact = math.sqrt(5)/2 + math.asinh(2)/4      # 公式解

def polyline_length(n):
    """n 段內接折線的總長"""
    xs = [a + i*(b-a)/n for i in range(n+1)]
    return sum(math.hypot(xs[i+1]-xs[i], f(xs[i+1])-f(xs[i])) for i in range(n))

print(f"{'n':>7} {'折線長':>14} {'誤差(低估)':>14} {'誤差比值':>10}")
prev = None
for n in [1, 2, 4, 8, 16, 32, 64, 128]:
    L = polyline_length(n)
    e = exact - L                              # 恆正 → 確實低估
    r = f"{prev/e:10.2f}" if prev else "         -"
    print(f"{n:7d} {L:14.9f} {e:14.3e} {r}")
    prev = e
print(f"{'公式解':>7} {exact:14.9f}")

# --- 樓梯逼近:形狀收斂,長度不收斂 ---
print("\n樓梯逼近直線 y=x 從 (0,0) 到 (1,1):")
for n in [2, 10, 100, 1000]:
    stair = n * (1/n) + n * (1/n)              # n 段水平 + n 段鉛直
    print(f"  n={n:5d}  樓梯總長 = {stair:.6f}   真正的斜邊 = {math.sqrt(2):.6f}")
print("  → 樓梯長度恆為 2,永遠不會收斂到 sqrt(2) ≈ 1.414")
print("    形狀看起來越來越像,長度卻差 41%。內接弦才會收斂。")

In [ ]:
# TODO 學生練習:把 f 換成 math.sin,區間 [0, pi],真值約 3.8201978
# 折線收斂的比值還是 4 嗎?要 n 多大才能對到小數第 4 位?

## Lab 2｜弧長參數化:讓動畫真正等速

觀念 6 說用 $t$ 當參數會在陡的地方「衝很快」。這格建一張弧長對照表,做出真正等距的取樣點——這就是圖學裡的做法。


In [ ]:
from scipy.integrate import quad
from scipy.optimize import brentq

f  = lambda t: t**2
df = lambda t: 2*t
a, b = 0.0, 2.0

def s_of_x(x):
    """從 a 走到 x 的弧長"""
    return quad(lambda t: math.sqrt(1 + df(t)**2), a, x)[0]

total = s_of_x(b)
print(f"曲線 y = x^2 在 [0,2] 的總弧長 = {total:.6f}")

# --- 均勻取 x:在陡的地方點會拉開 ---
N = 9
xs_uniform = np.linspace(a, b, N)
gaps_u = [math.hypot(xs_uniform[i+1]-xs_uniform[i],
                     f(xs_uniform[i+1])-f(xs_uniform[i])) for i in range(N-1)]

# --- 均勻取 s:反解 x(s),每段弧長相等 ---
targets = np.linspace(0, total, N)
xs_arc = [brentq(lambda x, S=S: s_of_x(x) - S, a, b) if 0 < S < total else (a if S <= 0 else b)
          for S in targets]
gaps_a = [math.hypot(xs_arc[i+1]-xs_arc[i], f(xs_arc[i+1])-f(xs_arc[i])) for i in range(N-1)]

print(f"\n{'':>18}{'最短段':>10}{'最長段':>10}{'長短比':>10}")
print(f"{'均勻取 x':>18}{min(gaps_u):10.4f}{max(gaps_u):10.4f}{max(gaps_u)/min(gaps_u):10.2f}")
print(f"{'均勻取弧長 s':>18}{min(gaps_a):10.4f}{max(gaps_a):10.4f}{max(gaps_a)/min(gaps_a):10.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
xs = np.linspace(a, b, 300)
for k, (pts, name) in enumerate([(xs_uniform, 'uniform in x'), (xs_arc, 'uniform in arc length')]):
    ax[k].plot(xs, f(xs), 'C0', lw=1.5)
    ax[k].plot(pts, [f(p) for p in pts], 'ro', ms=7)
    ax[k].set_title(name)
plt.tight_layout(); plt.show()
print("\n→ 左圖的點在陡處被拉開(動畫會『衝』);右圖沿曲線等距(真正等速)。")

In [ ]:
# TODO 學生練習:把 f 換成 lambda t: t**3,區間 [0,1.5]
# 均勻取 x 的長短比會變大還是變小?為什麼?